# Fish Instance Segmentation — Dataset Prep & Training

This notebook consolidates the annotated data into a single training set (no train/val/test split) and trains a YOLO segmentation model.

**Source data:**
- Images: `Kona- Hawaii.v1-biodiversity.yolo26/{train,valid,test}/images` — frames extracted from `nelha-trimmed_mp4`, currently split across the three folders.
- Segmentation labels: `segmentation_first/labels/train` — polygon annotations (57 fish classes). Only frames that actually contain a fish have a label file here (447 currently).

**What this notebook does:**
1. Finds every frame that currently has a real polygon annotation (i.e. a fish).
2. Pulls in *only* those frames — into one consolidated `segmentation_first/images/train` + `segmentation_first/labels/train` set, with no split. Unannotated frames (including empty/no-fish ones) are left out entirely.
3. Rewrites `segmentation_first/data.yaml` to point at this consolidated set, with no val/test split.
4. Trains a YOLO11 segmentation model on it.

Re-run the prep cells any time you add more annotations — they're safe to re-run and will pick up whatever is currently labeled, removing anything that's no longer labeled.

In [ ]:
%pip install -q pyyaml numpy opencv-python matplotlib

In [ ]:
import os
import re
import shutil
from pathlib import Path

import yaml

ROOT = Path.cwd()

# Original Roboflow export: images split across train/valid/test, bbox labels (unused here).
RAW_DATASET_DIR = ROOT / "Kona- Hawaii.v1-biodiversity.yolo26"
RAW_IMAGE_DIRS = [RAW_DATASET_DIR / split / "images" for split in ("train", "valid", "test")]

# Existing polygon (instance segmentation) annotations — only frames that have a fish get a file here.
SEG_SOURCE_LABELS_DIR = ROOT / "segmentation_first" / "labels" / "train"
SEG_SOURCE_DATA_YAML = ROOT / "segmentation_first" / "data.yaml"

# Consolidated, unsplit training set this notebook builds.
DATASET_DIR = ROOT / "segmentation_first"
IMAGES_OUT_DIR = DATASET_DIR / "images" / "train"
LABELS_OUT_DIR = DATASET_DIR / "labels" / "train"

FRAME_PATTERN = re.compile(r"^(?P<prefix>.+)-(?P<idx>\d+)_jpg\.rf\.")

for p in (RAW_DATASET_DIR, SEG_SOURCE_LABELS_DIR, SEG_SOURCE_DATA_YAML):
    assert p.exists(), f"Expected path not found: {p}"

print("Raw image dirs:", [str(p) for p in RAW_IMAGE_DIRS])
print("Segmentation labels source:", SEG_SOURCE_LABELS_DIR)
print("Consolidated output dataset dir:", DATASET_DIR)

In [ ]:
def parse_frame_idx(filename: str):
    m = FRAME_PATTERN.match(filename)
    return int(m.group("idx")) if m else None


# Every frame that currently has a real polygon annotation. Non-empty files only — this
# directory doubles as LABELS_OUT_DIR, so on a re-run it could also contain stray empty
# files from an earlier version of this pipeline, which must NOT count as "labeled".
labeled_label_files = sorted(
    f.name for f in SEG_SOURCE_LABELS_DIR.glob("*.txt") if f.stat().st_size > 0
)
labeled_idxs = {parse_frame_idx(f) for f in labeled_label_files}
assert None not in labeled_idxs, "Found a label file that doesn't match the expected frame naming pattern"
assert labeled_idxs, "No annotated (non-empty) label files found"

print(f"Labeled (fish) frames: {len(labeled_idxs)}")

In [ ]:
# Map every frame index -> source image path, scanning across the old train/valid/test image folders.
idx_to_image_path = {}
duplicate_idxs = set()

for img_dir in RAW_IMAGE_DIRS:
    for img_path in img_dir.glob("*.jpg"):
        idx = parse_frame_idx(img_path.name)
        if idx is None:
            continue
        if idx in idx_to_image_path:
            duplicate_idxs.add(idx)
        idx_to_image_path[idx] = img_path

assert not duplicate_idxs, f"Frame index appears in more than one split, ambiguous: {sorted(duplicate_idxs)}"

wanted_idxs = sorted(labeled_idxs)
missing_images = [i for i in wanted_idxs if i not in idx_to_image_path]
assert not missing_images, f"Missing source images for labeled frame indices: {missing_images}"

print(f"Total source images found across train/valid/test: {len(idx_to_image_path)}")
print(f"All {len(wanted_idxs)} labeled frames have a source image: OK")

In [ ]:
# Build the consolidated training set: fish-annotated frames only.
#
# Images are symlinked (not copied) into segmentation_first/images/train to avoid duplicating
# data on disk. Real annotations in segmentation_first/labels/train are left completely
# untouched. Anything in the output dirs that ISN'T one of the currently-labeled frames
# (e.g. left over from a previous run) is removed, so the result always matches exactly
# what's labeled right now.

IMAGES_OUT_DIR.mkdir(parents=True, exist_ok=True)
LABELS_OUT_DIR.mkdir(parents=True, exist_ok=True)

wanted_image_names = {idx_to_image_path[idx].name for idx in wanted_idxs}
wanted_label_names = {f"{idx_to_image_path[idx].stem}.txt" for idx in wanted_idxs}

linked_images = 0
for idx in wanted_idxs:
    src_image = idx_to_image_path[idx]
    dst_image = IMAGES_OUT_DIR / src_image.name
    if dst_image.is_symlink() or dst_image.exists():
        if not dst_image.is_symlink() or dst_image.resolve() != src_image.resolve():
            dst_image.unlink()
            dst_image.symlink_to(src_image)
    else:
        dst_image.symlink_to(src_image)
    linked_images += 1

# Drop any image symlink that isn't a currently-labeled frame (e.g. background frames from
# an earlier version of this pipeline).
removed_images = 0
for img_path in IMAGES_OUT_DIR.glob("*.jpg"):
    if img_path.name not in wanted_image_names:
        img_path.unlink()
        removed_images += 1

# Drop any empty (background) label file that isn't a real annotation.
removed_labels = 0
for label_path in LABELS_OUT_DIR.glob("*.txt"):
    if label_path.name not in wanted_label_names and label_path.stat().st_size == 0:
        label_path.unlink()
        removed_labels += 1

print(f"Images linked into {IMAGES_OUT_DIR}: {linked_images}")
print(f"Stale (non-labeled) images removed: {removed_images}")
print(f"Stale (empty/background) label files removed: {removed_labels}")
print(f"Total label files now in {LABELS_OUT_DIR}: {len(list(LABELS_OUT_DIR.glob('*.txt')))}")

In [ ]:
# Rewrite data.yaml: drop the train/val/test split, point everything at the consolidated set.
# No held-out val for now (per your call — you'll add more annotations before splitting out val/test).
with open(SEG_SOURCE_DATA_YAML) as f:
    existing_cfg = yaml.safe_load(f)

names = existing_cfg["names"]

new_cfg = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/train",  # placeholder: same as train since there's no val split yet
    "names": names,
}

with open(SEG_SOURCE_DATA_YAML, "w") as f:
    yaml.safe_dump(new_cfg, f, sort_keys=False, allow_unicode=True)

print(f"Wrote {SEG_SOURCE_DATA_YAML}:\n")
print(SEG_SOURCE_DATA_YAML.read_text())

## Sanity check

Spot-check a few frames with their polygons drawn on top before training on them.

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np


def load_polygons(label_path: Path):
    polygons, class_ids = [], []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        class_ids.append(int(parts[0]))
        coords = list(map(float, parts[1:]))
        polygons.append(np.array(coords).reshape(-1, 2))
    return class_ids, polygons


def show_frame(ax, idx: int):
    image_path = IMAGES_OUT_DIR / idx_to_image_path[idx].name
    label_path = LABELS_OUT_DIR / f"{idx_to_image_path[idx].stem}.txt"

    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    class_ids, polygons = load_polygons(label_path)

    ax.imshow(img)
    for cid, poly in zip(class_ids, polygons):
        poly_px = poly * [w, h]
        ax.plot(*np.vstack([poly_px, poly_px[0]]).T, linewidth=2)
        ax.text(*poly_px[0], names[cid], color="yellow", fontsize=8)
    ax.set_title(f"frame {idx}")
    ax.axis("off")


sample = random.sample(wanted_idxs, 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, idx in zip(axes, sample):
    show_frame(ax, idx)
plt.tight_layout()
plt.show()

## Train

- Model: `yolo26n-seg.pt` (YOLO26 nano segmentation, COCO-pretrained). Bump to `yolo26s/m/l/x-seg.pt` if you have more GPU budget.
- `imgsz=1280`: your frames are all `1280x720`, so this trains at native resolution — nothing gets downscaled. Ultralytics letterboxes the 720 side up to a 1280x1280 canvas rather than shrinking the 1280 side down.
- `batch=4`: **explicit, not auto.** Ultralytics' `batch=-1` autobatch only profiles on CUDA — on Apple's MPS backend (`device: mps`, which is what your run used) it silently falls back to a large default batch instead of sizing to your actual memory, which is exactly what caused the `MPS backend out of memory` crash at `imgsz=1280` on a 16GB Mac. Segmentation masks at native 1280px are memory-hungry, so start low: try `batch=4` first, and only raise it (6, 8, ...) if that trains cleanly. If you hit OOM again, **restart the kernel first** — a crashed MPS run can leave memory pinned in the process even after the exception, so the next attempt starts from an already-full pool.
- `val=False` skips validation entirely during training, since `val` in `data.yaml` currently points at the same images as `train` — running validation against training data would just give misleadingly optimistic metrics. Once you've annotated more frames and split out a real val set, update `data.yaml` and drop `val=False` here to get proper metrics/early stopping back.

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-seg.pt")  # nano seg model; bump to yolo26s/m/l/x-seg.pt if you have the GPU budget

results = model.train(
    data=str(SEG_SOURCE_DATA_YAML),
    epochs=100,
    imgsz=1280,  # native resolution (frames are 1280x720) — no downscaling
    batch=4,     # explicit — see note above on why batch=-1 isn't safe on MPS
    val=False,
    project="runs/segment",
    name="fish_seg",
)